# 09 — Diagnóstico de `rollout_m` (temporal ensembling)

Notebook separado do `08_avaliar_arquitetura.ipynb` -- aqui o objetivo não é
medir taxa de sucesso da arquitetura, e sim **comparar valores de `m`** do
temporal ensembling (`eval/rollout_libero.py::temporal_ensemble`) sobre
cenários que JA SE SABE que falharam, pra ver se algum valor recupera melhor
de um erro do que o `rollout_m` padrão do config.

$$w_i = e^{-m \cdot i} \qquad \hat{w}_i = \frac{w_i}{\sum_j w_j} \qquad a_t = \sum_i \hat{w}_i \, a_i$$

com $i=0$ a predição MAIS ANTIGA (favorece suavidade, `m>0`) e $i=L-1$ a MAIS
NOVA (favorece reatividade a erro, `m<0`).

**Pré-requisito**: já ter rodado uma avaliação completa da arquitetura em
`08_avaliar_arquitetura.ipynb` (seções 5 e 9), com o JSON de resultados salvo
em `CHECKPOINT_DIR/resultados/`. Este notebook carrega esse JSON pra achar
cenários que falharam de verdade, em vez de rodar o rollout completo de novo
só pra descobrir isso.

**Segundo eixo, além de `m`**: `MAX_STEPS_DIAG` (seção 6) -- dá mais passos
que o padrão do treino (`cfg["rollout_max_steps"]`) pra política ter mais
tempo de tentar se recuperar antes do episódio ser cortado por timeout.

Precisa de `lerobot[libero]` (simulador de verdade) além de `hdf5`/`language`/`vlm`.

## 1. Repositório e ambiente

In [ ]:
!git clone -b teste https://github.com/rafaelheydt/act-lang.git
%cd act-lang

!pip install -q -e ".[hdf5,language,vlm]" "lerobot[libero]"

import sys
sys.path.insert(0, "src")
sys.path.insert(0, ".")
sys.path.insert(0, "scripts")

## 2. Escolha de arquitetura

MESMA escolha usada no treino/avaliação original -- tem que bater com o
checkpoint e com o JSON de resultados que você quer diagnosticar.

In [ ]:
BACKBONE_TYPE = "clip_vitb32"  # "resnet18" | "clip_vitb32" | "siglip2_base"
FUSION_TYPE = "film"           # "token" | "film" | "cross_attn"

_CLI_CONFIG_NAME = {
    ("resnet18", "token"): "language_token",
    ("resnet18", "film"): "language_film",
    ("resnet18", "cross_attn"): "language_cross_attn",
    ("clip_vitb32", "film"): "language_clip_film",
    ("clip_vitb32", "token"): "language_clip_token",
    ("clip_vitb32", "cross_attn"): "language_clip_cross_attn",
    ("siglip2_base", "film"): "language_siglip_film",
    ("siglip2_base", "token"): "language_siglip_token",
    ("siglip2_base", "cross_attn"): "language_siglip_cross_attn",
}
CLI_CONFIG_NAME = _CLI_CONFIG_NAME[(BACKBONE_TYPE, FUSION_TYPE)]

import train as train_script
cfg = train_script.load_config(CLI_CONFIG_NAME)
print(f"--config {CLI_CONFIG_NAME!r} -> experiment_name={cfg['experiment_name']!r}")
print(f"task_suite_name={cfg['task_suite_name']!r} | rollout_m padrão={cfg['rollout_m']} | rollout_max_steps padrão={cfg['rollout_max_steps']}")

## 3. Normalizadores (mesmo subconjunto de tarefas do treino)

`TASK_LIMIT` precisa bater com o que foi usado no treino -- decide quais
tarefas foram treinadas E reproduz os stats exatos de normalização.

In [ ]:
TASK_LIMIT = 2  # mesmo valor usado no treino -- None se treinou nas 10 completas
DATA_DIR = "/content/libero_hdf5_object"

import subprocess
cmd = [sys.executable, "-u", "scripts/download_libero_hdf5.py", "--out", DATA_DIR, "--suite", "libero_object"]
if TASK_LIMIT is not None:
    cmd += ["--limit", str(TASK_LIMIT)]
print("rodando:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, "download falhou -- veja o log acima"

import torch
from act_lang.data.hdf5_libero import load_meta, build_normalizers_from_meta
from act_lang.utils.runtime import describe_devices, pick_device, is_colab

print(describe_devices())
device = pick_device(preferred_index=cfg.get("device_index"))
print(f"usando: {device}")

meta = load_meta(DATA_DIR)
state_norm, action_norm = build_normalizers_from_meta(meta, device)

from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
lerobot_meta = LeRobotDatasetMetadata("lerobot/libero")
video_fps = int(lerobot_meta.fps)
print(f"normalizadores prontos | video_fps={video_fps}")

## 4. Carregar o checkpoint treinado

`train_script.build_model_and_optimizer` resolve o backbone/fusão/texto
pareado certo a partir do MESMO `cfg` do treino.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
CHECKPOINT_DIR = Path(f"/content/drive/MyDrive/{cfg['experiment_name']}_hdf5128")
print("checkpoints em:", CHECKPOINT_DIR)

from act_lang.training.checkpoints import load_checkpoint

model, _optimizer = train_script.build_model_and_optimizer(cfg, device)
best = sorted(CHECKPOINT_DIR.glob("best_epoch*.pt"))[-1]  # ou aponte o arquivo à mão
print(f"carregando: {best.name}")
next_epoch, _ = load_checkpoint(best, model, device=device)
model.eval()
print(f"checkpoint da época {next_epoch - 1}")

## 5. Carregar cenários que falharam (JSON de `08_avaliar_arquitetura.ipynb`)

Em vez de rodar o rollout completo de novo só pra achar episódios que
falharam, reaproveita o JSON já exportado por `08_avaliar_arquitetura.ipynb`
(seção 9, `CHECKPOINT_DIR/resultados/resultados_*.json`). Pega o mais
recente por padrão -- ajuste `json_path` na mão se quiser um específico.

In [ ]:
import json

resultados_dir = CHECKPOINT_DIR / "resultados"
json_path = sorted(resultados_dir.glob("resultados_*.json"))[-1]
with open(json_path, encoding="utf-8") as f:
    payload = json.load(f)

assert payload["backbone_type"] == BACKBONE_TYPE and payload["fusion_type"] == FUSION_TYPE, (
    f"JSON é de {payload['backbone_type']}+{payload['fusion_type']}, "
    f"mas a sessão está configurada para {BACKBONE_TYPE}+{FUSION_TYPE}"
)

all_results = payload["por_tarefa"]
print(f"carregado de {json_path.name}: {len(all_results)} tarefa(s), "
      f"taxa geral = {payload['taxa_sucesso_geral']:.1%}")

falhas = [
    (task_text, r["init_state_id"])
    for task_text, results in all_results.items()
    for r in results
    if not r["success"]
]
assert falhas, (
    "nenhum episódio falhou nesse JSON -- não há cenário de erro conhecido pra "
    "testar recuperação. Rode/ache uma avaliação com alguma falha."
)
print(f"\n{len(falhas)} cenário(s) que falharam com m={cfg['rollout_m']}:")
for t, sid in falhas:
    print(f"  - {t!r} (init_state_id={sid})")

## 6. Configurar o diagnóstico

`M_VALUES`: valores de `m` a comparar (o 1º é o padrão do config, pra
referência). `MAX_STEPS_DIAG`: teto de passos por episódio -- maior que o
padrão do treino dá mais chance da política se recuperar antes do timeout.
`CENARIOS`: quais falhas da seção 5 entram na comparação (por padrão, todas).

In [ ]:
M_VALUES = [cfg["rollout_m"], 0.0, -0.01, -0.02, -0.05]  # 1º valor = o padrão do config
MAX_STEPS_DIAG = cfg["rollout_max_steps"] * 2  # mais tempo pra tentar recuperar
CENARIOS = falhas  # ou uma fatia, ex.: falhas[:3], pra rodar menos combinações

print(f"M_VALUES = {M_VALUES}")
print(f"MAX_STEPS_DIAG = {MAX_STEPS_DIAG} (padrão do treino: {cfg['rollout_max_steps']})")
print(f"{len(CENARIOS)} cenário(s) x {len(M_VALUES)} valor(es) de m = {len(CENARIOS) * len(M_VALUES)} rollouts")

## 7. Rodar a grade cenário × m

Reusa o mesmo `env` entre os `m` de um mesmo cenário (só recria ao trocar de
tarefa, que é quando o `task_id` muda). Salva um vídeo por combinação em
`video_root/diagnostico_rollout_m/<tarefa>/<cenario>/<m>/`.

In [ ]:
from libero.libero import benchmark
from lerobot.envs.libero import LiberoEnv
from act_lang.eval.rollout_libero import rollout_libero

task_suite = benchmark.get_benchmark_dict()[cfg["task_suite_name"]]()
def _task_id(text):
    return next(i for i in range(len(task_suite.tasks)) if task_suite.get_task(i).language == text)

video_root = "/content/libero_rollouts" if is_colab() else str(CHECKPOINT_DIR / "rollouts")
diag_video_root = f"{video_root}/diagnostico_rollout_m"

diag_results = []  # lista de dicts: tarefa, init_state_id, m, success, steps
for task_text, init_state_id in CENARIOS:
    task_id = _task_id(task_text)
    cenario_slug = f"{task_text[:30].replace(' ', '_')}_state{init_state_id:02d}"
    print(f"\n=== {task_text!r} (init_state_id={init_state_id}) ===")
    env = LiberoEnv(
        task_suite=task_suite, task_id=task_id, task_suite_name=cfg["task_suite_name"],
        control_mode="relative", render_mode="rgb_array",
        obs_type="pixels_agent_pos",
        observation_width=128, observation_height=128,
    )
    for m in M_VALUES:
        label = f"m{m:+.3f}".replace("+", "p").replace("-", "n").replace(".", "_")
        r = rollout_libero(
            model, env, state_norm, action_norm, device,
            m=m, max_steps=MAX_STEPS_DIAG,
            video_dir=f"{diag_video_root}/{cenario_slug}/{label}",
            video_fps=video_fps, task_text=task_text,
            init_state_ids=[init_state_id],
        )[0]
        diag_results.append({
            "tarefa": task_text, "init_state_id": init_state_id,
            "m": m, "success": r["success"], "steps": r["steps"],
        })
        print(f"  m={m:+.3f} -> sucesso={r['success']} | steps={r['steps']}")
    env.close()

## 8. Comparar desempenho por `m`

In [ ]:
import pandas as pd

diag_df = pd.DataFrame(diag_results)

resumo_m = diag_df.groupby("m").agg(
    taxa_sucesso=("success", "mean"),
    steps_medio=("steps", "mean"),
    n_cenarios=("success", "count"),
).reset_index().sort_values("m", ascending=False)
print("resumo por m (sobre os cenários que falharam com o m padrão):")
print(resumo_m.to_string(index=False))

print("\ndetalhe por cenário:")
pivot = diag_df.pivot(index=["tarefa", "init_state_id"], columns="m", values="success")
print(pivot.to_string())

## 9. Salvar vídeos e exportar resultados no Drive

Mesmo padrão de `08_avaliar_arquitetura.ipynb` (seções 8-9), mas gravando
em `CHECKPOINT_DIR/diagnostico_rollout_m/` pra não misturar com os
resultados da avaliação principal.

In [ ]:
import shutil
from datetime import datetime

diag_dir = CHECKPOINT_DIR / "diagnostico_rollout_m"
diag_dir.mkdir(parents=True, exist_ok=True)

destino_videos = diag_dir / "videos"
destino_videos.mkdir(parents=True, exist_ok=True)
shutil.copytree(diag_video_root, destino_videos, dirs_exist_ok=True)
n_videos = len(list(destino_videos.rglob("*.mp4")))
print(f"{n_videos} vídeo(s) copiados para {destino_videos}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
csv_path = diag_dir / f"diagnostico_{timestamp}.csv"
diag_df.to_csv(csv_path, index=False)

json_out_path = diag_dir / f"diagnostico_{timestamp}.json"
with open(json_out_path, "w", encoding="utf-8") as f:
    json.dump({
        "backbone_type": BACKBONE_TYPE, "fusion_type": FUSION_TYPE,
        "m_values": M_VALUES, "max_steps_diag": MAX_STEPS_DIAG,
        "fonte_falhas": str(json_path.name),
        "resultados": diag_results,
    }, f, ensure_ascii=False, indent=2)

print(f"resumo (CSV): {csv_path}")
print(f"resultado completo (JSON): {json_out_path}")

## 10. Ver um cenário, todos os `m`, lado a lado

Escolha o cenário (índice em `CENARIOS`) pra comparar visualmente.

In [ ]:
from IPython.display import HTML, Video, display

CENARIO_IDX = 0  # índice em CENARIOS
task_text, init_state_id = CENARIOS[CENARIO_IDX]
cenario_slug = f"{task_text[:30].replace(' ', '_')}_state{init_state_id:02d}"

for m in M_VALUES:
    row = next(d for d in diag_results if d["tarefa"] == task_text and d["init_state_id"] == init_state_id and d["m"] == m)
    label = f"m{m:+.3f}".replace("+", "p").replace("-", "n").replace(".", "_")
    tag = "sucesso" if row["success"] else "falha"
    display(HTML(f"<b>m={m:+.3f}</b> -- {tag}, {row['steps']} passos"))
    display(Video(f"{diag_video_root}/{cenario_slug}/{label}/ep00_state{init_state_id:02d}_{tag}.mp4", embed=True, width=500))